# Simple RAG with Filtering + Query Transformations

## 적용 기법
- **Metadata Filtering** : cate_depth1 기반 장르 필터
- **Query Transformations** : Sub-query Decomposition → 서브쿼리별 검색결합

## 0. 설치 및 초기화

In [ ]:
!git clone https://github.com/jjeong3150/AIFFEL_final_pjt_book-recommendation-agent

fatal: destination path 'AIFFEL_final_pjt_book-recommendation-agent' already exists and is not an empty directory.


In [ ]:
!pip install -q qdrant-client
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-naver

In [ ]:
%run /content/AIFFEL_final_project_peekabook/research/src/state/state_v2.ipynb
%run /content/AIFFEL_final_project_peekabook/research/src/db/qdrant.py
%run /content/AIFFEL_final_project_peekabook/research/src/embedding/embedder.py
%run /content/AIFFEL_final_project_peekabook/research/src/reranking/reranker.py

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"]    = userdata.get('OPENAI_API_KEY')
os.environ["CLOVASTUDIO_API_KEY"] = userdata.get('CLOVASTUDIO_API_KEY')
os.environ["QDRANT_API_KEY"]    = userdata.get('QDRANT_API_KEY')
os.environ["QDRANT_URL"]        = userdata.get('QDRANT_URL')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_naver import ChatClovaX

embedder = LocalEmbedder("BAAI/bge-m3")
db       = QdrantDB(vector_size=1024)
llm      = ChatOpenAI(model="gpt-4o-mini")
reranker = LocalReranker("BAAI/bge-reranker-v2-m3")
# llm    = ChatClovaX(model="HCX-005", temperature=0)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[LocalReranker] 캐시 로드: /content/drive/MyDrive/aiffel_final_pjt/models/BAAI_bge-reranker-v2-m3


## 1. 장르 추출 노드 (simple_rag_with_filtering과 동일)

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from qdrant_client.models import Filter, FieldCondition, MatchAny
import json

CATEGORY_LIST = ["소설", "대학교재/전문서적", "어린이", "수험서/자격증", "시/에세이", "종교", "유아", "만화",
                 "사회/정치", "경제/경영", "인문", "예술/대중문화", "국어/외국어", "고등학교 참고서",
                 "자기계발", "초등학교 참고서", "건강/취미", "컴퓨터/IT", "역사", "자연/과학",
                 "청소년", "중학교 참고서", "가정/요리", "여행", "잡지", "전집", "외국도서"]

genre_prompt = ChatPromptTemplate.from_template("""
사용자 프로파일을 보고 아래 카테고리 목록에서 적합한 것을 최대 3개 선택하세요.
목록에 없는 값은 절대 반환하지 마세요.

카테고리 목록: {category_list}
사용자 프로파일: {summary}

JSON으로만 반환: {{"categories": ["소설"]}}
""")

def extract_genre_node(state: GraphState) -> dict:
    summary = state.get("summary", "")
    chain = genre_prompt | llm
    response = chain.invoke({
        "category_list": CATEGORY_LIST,
        "summary": summary
    })
    try:
        categories = json.loads(response.content)["categories"]
    except (json.JSONDecodeError, KeyError):
        categories = []
    print(f"추출된 장르: {categories}")
    return {"genre_filter": categories}

## 2. Step-back Prompting
**입력**: 원본 사용자 프로파일  
**출력**: 추상화된 독서 목적  
**목적**: 구체적인 프로파일에서 한 단계 물러나 독자 타겟·독서 목적을 추상화 → 의도 파악 문제 해결

In [ ]:
step_back_prompt = ChatPromptTemplate.from_template("""
당신은 도서 추천 시스템의 AI 어시스턴트입니다.
아래 사용자 프로파일에서 한 단계 물러나,
더 넓은 범위의 도서를 검색할 수 있는 일반적인 쿼리를 생성하세요.
특정 장르나 조건에 국한되지 않고 독서 경험과 목적 중심으로 작성하세요.
2문장 이내로 작성하세요.

사용자 프로파일: {summary}

출력:
""")

def step_back_query(summary: str, llm) -> str:
    """
    사용자 프로파일 → 추상화된 독서 목적

    Args:
        summary : 원본 사용자 프로파일
        llm     : LLM 인스턴스

    Returns:
        str: 추상화된 독서 목적
    """
    chain = step_back_prompt | llm
    return chain.invoke({"summary": summary}).content.strip()

## 3. Query Rewriting
**입력**: step_back 결과 (추상화된 독서 목적)  
**출력**: 검색에 최적화된 구체적인 쿼리  
**목적**: 추상화된 의도를 도서 검색에 더 적합한 표현으로 구체화

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_template("""
당신은 도서 추천 시스템의 AI 어시스턴트입니다.
아래 독서 목적을 도서 검색에 더 적합하고 구체적인 검색어로 재작성하세요.
장르, 독자 수준, 분위기 등 검색 정확도를 높일 수 있는 표현을 포함하세요.

독서 목적: {step_back}

재작성된 검색 쿼리 (두 문장 이내로):
""")

def rewrite_query(step_back: str, llm) -> str:
    """
    추상화된 독서 목적 → 검색에 최적화된 구체적 쿼리

    Args:
        step_back : step_back_query()의 결과
        llm       : LLM 인스턴스

    Returns:
        str: 재작성된 검색 쿼리
    """
    chain = rewrite_prompt | llm
    return chain.invoke({"step_back": step_back}).content.strip()

## 4. Sub-query Decomposition
**입력**: rewritten 결과 (구체화된 쿼리)  
**출력**: 조건 단위로 분해된 서브쿼리 리스트  
**목적**: 다중 조건을 분해하여 각각 검색 → RRF 결합으로 조건 만족률 향상

In [ ]:
decompose_prompt = ChatPromptTemplate.from_template("""
당신은 도서 추천 시스템의 검색 쿼리 전문가입니다.
사용자의 독서 취향과 상황을 깊이 이해하여, 벡터 임베딩 검색에 최적화된 서브쿼리를 생성합니다.

아래 검색 쿼리를 2~4개의 서브쿼리로 분해하세요.
각 서브쿼리는 독립적인 문장으로 작성하되, 전체적으로 자연스럽게 맥락이 이어지도록 하세요.
원래 쿼리의 조건을 충실히 반영하면서, 사용자가 미처 생각하지 못했을 관련 관점을 한 개 포함하세요.


[주의]
- "이 중에서", "그 중에서" 같은 참조 표현은 사용하지 마세요.
- 리뷰, 평점, 사용자 의견 등 도서 메타데이터 외의 정보를 요청하지 마세요.
- "추천 도서", "추천해주세요", "포함해주세요" 같은 표현으로 끝내지 마세요.

검색 쿼리: {rewritten}

출력 형식 (번호와 텍스트만, 다른 텍스트 없이):
1. [서브쿼리 1]
2. [서브쿼리 2]
3. [서브쿼리 3]
""")


def decompose_query(rewritten: str, llm) -> list:
    """
    구체화된 쿼리 → 서브쿼리 리스트

    Args:
        rewritten : rewrite_query()의 결과
        llm       : LLM 인스턴스

    Returns:
        List[str]: 서브쿼리 리스트
    """
    chain = decompose_prompt | llm
    response = chain.invoke({"rewritten": rewritten}).content
    sub_queries = [
        q.strip().lstrip("1234567890. ")
        for q in response.split("\n")
        if q.strip() and q.strip()[0].isdigit()
    ]
    return sub_queries

## 5. Chained Pipeline
세 단계를 순서대로 이어서 실행하는 메인 **함수**

In [ ]:
def get_chained_queries(summary: str, llm,
                        use_step_back: bool = True,
                        use_rewrite: bool = True,
                        use_decompose: bool = True) -> dict:
    """
    세 가지 변환을 순서대로 이어서 실행

    흐름: summary → step_back → rewritten → sub_queries

    Args:
        summary       : 원본 사용자 프로파일
        llm           : LLM 인스턴스
        use_step_back : Step-back Prompting 사용 여부 (기본 True)
        use_rewrite   : Query Rewriting 사용 여부 (기본 True)
        use_decompose : Sub-query Decomposition 사용 여부 (기본 True)

    Returns:
        {
            'step_back'  : str,       # Step-back 결과
            'rewritten'  : str,       # Rewriting 결과
            'sub_queries': List[str], # Decomposition 결과
            'all'        : List[str], # RRF에 넘길 전체 쿼리 목록
        }
    """
    # 1단계: Step-back → 의도 추상화
    step_back = step_back_query(summary, llm) if use_step_back else summary
    print(f"  [Step-back]  : {step_back}")

    # 2단계: Rewriting → 추상화된 의도를 구체적 검색어로
    rewritten = rewrite_query(summary, llm) if use_rewrite else summary
    print(f"  [Rewritten]  : {rewritten}")

    # 3단계: Decomposition → 구체적 쿼리를 조건 단위로 분해
    sub_queries = decompose_query(rewritten, llm) if use_decompose else []
    print(f"  [Sub-queries]: {sub_queries}")

    # 전체 쿼리 목록 (RRF 입력용)
    all_queries = [step_back, rewritten] + sub_queries

    return {
        "step_back":   step_back,
        "rewritten":   rewritten,
        "sub_queries": sub_queries,
        "all":         all_queries,
    }

## 6. 테스트 실행

In [ ]:
# # TEST_SUMMARY = "사용자는 한국 현대소설을 선호하며, 감성을 풍부하게 하고 다양한 시각을 배우고자 하는 목표를 가지고 있습니다. 현재 감성적이고 몽환적인 기분 속에서 잔잔한 음악과 함께 조용한 공간에서 독서를 즐기고 있으며, 스토리와 감정에 집중하는 스타일을 선호합니다. 중급 난이도의 작품을 통해 감정 정리와 아름다움을 느끼고자 하는 상황입니다."
# TEST_SUMMARY = "사용자는 주말에 가볍게 읽으면서 스트레스를 해소하고 새로운 아이디어를 얻고자 하며, SF와 테크 스릴러 장르의 흥미로운 이야기를 선호합니다. 독서 스타일은 빠르게 전체적인 흐름을 파악하고, 스토리의 흡입력에 집중하는 중급자용입니다. 깊이 있는 분석보다는 이야기 전개와 흥미로운 설정에 중점을 두고 있으며, 가벼운 책을 통해 재미를 추구하고 있습니다."

# print("[Query Transformations 실행]")
# queries = get_chained_queries(TEST_SUMMARY, llm)

# print(f"\n[전체 쿼리 목록] 총 {len(queries['all'])}개")
# for i, q in enumerate(queries['all'], 1):
#     print(f"  {i}. {q}")

[Query Transformations 실행]
  [Step-back]  : 주말 동안의 스트레스를 해소하고 새로운 아이디어를 얻을 수 있는 흥미롭고 몰입감 있는 소설을 찾아보세요. 빠르게 읽을 수 있는 이야기 전개와 매력적인 설정이 돋보이는 도서를 추천합니다.
  [Rewritten]  : "중급자를 위한 가벼운 SF 및 테크 스릴러 소설, 이야기 전개와 흥미로운 설정에 중점을 두어 빠르게 읽을 수 있는 작품을 추천해 주세요. 스트레스 해소와 새로운 아이디어 발상을 도와줄 흥미로운 이야기 중심의 도서를 찾고 있습니다."
  [Sub-queries]: ['중급자를 위한 가벼운 SF 소설로, 이야기 전개와 흥미로운 설정에 중점을 두고 빠르게 읽을 수 있는 작품을 찾아보세요.', '테크 스릴러 장르에서 스트레스 해소와 새로운 아이디어 발상을 도와줄 흥미로운 이야기 중심의 도서를 추천합니다.', '빠르게 읽을 수 있으면서도 상상력을 자극하는 독창적인 설정의 작품이 있다면, 그에 대한 정보도 확인해 보세요.']

[전체 쿼리 목록] 총 5개
  1. 주말 동안의 스트레스를 해소하고 새로운 아이디어를 얻을 수 있는 흥미롭고 몰입감 있는 소설을 찾아보세요. 빠르게 읽을 수 있는 이야기 전개와 매력적인 설정이 돋보이는 도서를 추천합니다.
  2. "중급자를 위한 가벼운 SF 및 테크 스릴러 소설, 이야기 전개와 흥미로운 설정에 중점을 두어 빠르게 읽을 수 있는 작품을 추천해 주세요. 스트레스 해소와 새로운 아이디어 발상을 도와줄 흥미로운 이야기 중심의 도서를 찾고 있습니다."
  3. 중급자를 위한 가벼운 SF 소설로, 이야기 전개와 흥미로운 설정에 중점을 두고 빠르게 읽을 수 있는 작품을 찾아보세요.
  4. 테크 스릴러 장르에서 스트레스 해소와 새로운 아이디어 발상을 도와줄 흥미로운 이야기 중심의 도서를 추천합니다.
  5. 빠르게 읽을 수 있으면서도 상상력을 자극하는 독창적인 설정의 작품이 있다면, 그에 대한 정보도 확인해 보세요.


## 7. Ablation Study용 — 단계별 조합 테스트

In [ ]:
# # Sub-query만 사용
# print("[Sub-query only]")
# q1 = get_chained_queries(TEST_SUMMARY, llm,
#                           use_step_back=False,
#                           use_rewrite=False,
#                           use_decompose=True)

# # Step-back + Sub-query
# print("\n[Step-back + Sub-query]")
# q2 = get_chained_queries(TEST_SUMMARY, llm,
#                           use_step_back=True,
#                           use_rewrite=False,
#                           use_decompose=True)

# # 전체 (Step-back + Rewriting + Sub-query)
# print("\n[전체 파이프라인]")
# q3 = get_chained_queries(TEST_SUMMARY, llm,
#                           use_step_back=True,
#                           use_rewrite=True,
#                           use_decompose=True)

## 8. RRF 함수 정의

In [ ]:
def reciprocal_rank_fusion(results_list: list, k: int = 60) -> list:
    """여러 검색 결과를 RRF로 결합. Returns: payload 딕셔너리 리스트 (점수 내림차순)"""
    scores, payloads = {}, {}
    for results in results_list:
        for rank, r in enumerate(results):
            isbn = r.payload.get("isbn", "")
            if isbn:
                scores[isbn]   = scores.get(isbn, 0) + 1 / (k + rank + 1)
                payloads[isbn] = r.payload
    return [payloads[isbn] for isbn, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

## 9. Query Transform RAG 노드

In [ ]:
def query_transform_rag_node(state: GraphState) -> dict:
    summary    = state.get("summary", "")
    categories = state.get("genre_filter", [])

    # ── 1. Query Transformations ────────────────────────────
    print("\n[Query Transformations]")
    queries     = get_chained_queries(summary, llm)
    all_queries = queries["all"]

    # ── 2. 장르 필터 구성 (simple_rag_with_filtering과 동일) ──
    query_filter = None
    if categories:
        query_filter = Filter(
            must=[FieldCondition(key="cate_depth1", match=MatchAny(any=categories))]
        )

    # ── 3. 쿼리별 검색 ──────────────────────────────────────
    all_results = []
    for query in all_queries:
        query_vector = embedder.embed(query)
        if query_filter:
            results = db.search_with_filter(
                "books_v1", query_vector,
                query_filter=query_filter, limit=5, threshold=0.5,
            )
        else:
            results = db.search("books_v1", query_vector, limit=5, threshold=0.5)
        all_results.append(results)

        # print(query)
        # for r in results:
        #     print(f"score: {r.score}")
        #     print(f"title: {r.payload.get('title')}")
        #     print(f"book_intro: {r.payload.get('book_intro')}")
        # print("---------------")
    # ── 4. RRF 결합
    merged_payloads = reciprocal_rank_fusion(all_results)

    # ── 5. Reranker Model 적용
    reranked_payloads = reranker.rerank(query=summary, books=merged_payloads)

    # ── 6. 결과 정리
    retrieved_books = [
        {
            "isbn":        p.get("isbn"),
            "title":       p.get("title"),
            "author":      p.get("author"),
            "book_intro":  p.get("book_intro"),
            "cate_depth1": p.get("cate_depth1"),
        }
        for p in reranked_payloads[:5]
    ]

    # print(f"\n최종 검색 결과: {len(retrieved_books)}권")
    return {"retrieved_books": retrieved_books}

## 10. LLM 노드

*   항목 추가
*   항목 추가



In [ ]:
def rag_llm_node(state: GraphState) -> dict:
    summary = state.get("summary", "")
    books   = state["retrieved_books"]

    context = "\n\n".join([
        f"ISBN: {b['isbn']}\n"
        f"제목: {b['title']}\n"
        f"저자: {b['author']}\n"
        f"장르: {b['cate_depth1']}\n"
        f"소개: {b['book_intro'][:100]}..."
        for b in books
    ])

    rag_prompt = ChatPromptTemplate.from_template("""
당신은 도서관 큐레이터 AI입니다.

[규칙]
- 반드시 [검색된 도서 목록]에 있는 책만 추천하세요.
- 반드시 JSON 형식으로만 답하세요. 다른 텍스트는 절대 포함하지 마세요.
- 사용자 프로파일을 참고해서 가장 적합한 도서 3권을 추천하세요.
- 추천 이유는 반드시 사용자 프로파일의 독서 목적, 성향, 상황과 연결해서 작성하세요.
- 장르는 반드시 [검색된 도서 목록]의 장르 값을 그대로 사용하세요.

[사용자 프로파일]
{summary}

[검색된 도서 목록]
{context}

[출력 형식]
[
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "cate_depth1": "장르", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "cate_depth1": "장르", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "cate_depth1": "장르", "reason": "추천 이유 2~3문장"}}
]
""")

    chain    = rag_prompt | llm
    response = chain.invoke({"context": context, "summary": summary})

    try:
        recommendations = json.loads(response.content)
    except json.JSONDecodeError:
        recommendations = response.content

    return {
        "messages":       [AIMessage(content=response.content)],
        "recommendations": recommendations
    }

## 11. 그래프 구성 및 실행

In [ ]:
# graph_test = StateGraph(GraphState)
# graph_test.add_node("genre", extract_genre_node)
# graph_test.add_node("rag",   query_transform_rag_node)  # ← 변경된 노드
# graph_test.add_node("llm",   rag_llm_node)
# graph_test.add_edge(START,   "genre")
# graph_test.add_edge("genre", "rag")
# graph_test.add_edge("rag",   "llm")
# graph_test.add_edge("llm",   END)
# app_test = graph_test.compile()

# # TEST_SUMMARY = "사용자는 한국 현대소설을 선호하며, 감성을 풍부하게 하고 다양한 시각을 배우고자 하는 목표를 가지고 있습니다. 현재 감성적이고 몽환적인 기분 속에서 잔잔한 음악과 함께 조용한 공간에서 독서를 즐기고 있으며, 스토리와 감정에 집중하는 스타일을 선호합니다. 중급 난이도의 작품을 통해 감정 정리와 아름다움을 느끼고자 하는 상황입니다."
# TEST_SUMMARY = "사용자는 주말에 가볍게 읽으면서 스트레스를 해소하고 새로운 아이디어를 얻고자 하며, SF와 테크 스릴러 장르의 흥미로운 이야기를 선호합니다. 독서 스타일은 빠르게 전체적인 흐름을 파악하고, 스토리의 흡입력에 집중하는 중급자용입니다. 깊이 있는 분석보다는 이야기 전개와 흥미로운 설정에 중점을 두고 있으며, 가벼운 책을 통해 재미를 추구하고 있습니다."
# # TEST_SUMMARY = "사용자는 수업에서 활용할 수 있는 흥미로운 역사 이야기나 사례를 찾고 있으며, 역사 비문학 장르를 선호합니다. 천천히 깊이 있게 읽는 스타일로, 중요한 내용이나 흥미로운 부분을 메모하여 학생들에게 설명하는 데 활용하려고 합니다. 중급 난이도의 책을 선호하며, 학생들에게 긍정적인 영향을 줄 수 있는 내용을 중시하고 있습니다."

# result_test = app_test.invoke({
#     "messages":        [HumanMessage(content=TEST_SUMMARY)],
#     "summary":         TEST_SUMMARY,
#     "retrieved_books": [],
#     "recommendations": [],
#     "genre_filter":    [],
# })

# print(result_test["messages"][-1].content)

추출된 장르: ['소설']

[Query Transformations]
  [Step-back]  : 주말에 가볍게 즐길 수 있는 흥미로운 스토리가 담긴 도서를 추천해 주세요. 독자의 호기심을 자극하고 새로운 아이디어를 제공할 수 있는 다양한 장르의 책들을 포함해 주시면 좋겠습니다.
  [Rewritten]  : "주말 가벼운 독서를 위한 중급자용 SF 및 테크 스릴러 소설, 흥미로운 설정과 스토리 전개에 집중하며 스트레스 해소와 새로운 아이디어를 자극할 수 있는 책 찾기."
  [Sub-queries]: ['주말 가벼운 독서를 위해 중급자용 SF 소설을 찾아보세요.', '테크 스릴러 장르에서 흥미로운 설정과 스토리 전개를 가진 도서를 추천합니다.', '스트레스 해소와 새로운 아이디어 자극에 도움이 되는 책들도 고려해보면 좋습니다.']

[Reranking 결과]
  score: 0.1352 | 숨겨진 얼굴
  score: 0.1047 | 영원한 전쟁 - 폰 북스 9
  score: 0.0856 | 타임리미트 2
  score: 0.0359 | 노인과 바다 - 토론을 위한 세계명작 1
  score: 0.0199 | 포스텍 SF 어워드 수상작품집 No.1
  score: 0.0166 | 체포되셨습니다 2 - 홍염의 꽃
  score: 0.0133 | 여행 그리고 파리 - Travel and Paris
  score: 0.0130 | 블루스카이 3 - 피로 얼룩진 혁명군
  score: 0.0123 | 당신의 머리 위에 1
  score: 0.0103 | 무채색
[
    {
        "title": "영원한 전쟁 - 폰 북스 9",
        "author": "조 홀드먼",
        "isbn": "9788972594345",
        "book_intro": "SF의고전.면작들을 엄선하여 깔끔한 번역으로 선보이는 SF 시리즈물이다. 국내에는 과학소설의 계층이 얇아 SF소설이 활성화되지 않았다. 그런와중에서도 시공 그리폰 북스 는 SF 매

## DeepEval 평가

In [ ]:
# import logging
# logging.getLogger("deepeval").setLevel(logging.WARNING)

# from deepeval import evaluate
# from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
# from deepeval.test_case import LLMTestCase

# test_cases = [
#     LLMTestCase(
#         input=TEST_SUMMARY,
#         actual_output="\n".join([
#             f"[{b['title']}] {b['reason']}"
#             for b in result["recommendations"]
#         ]),
#         retrieval_context=[
#             f"[{b['title']}] {b['book_intro']}"
#             for b in result["retrieved_books"]
#         ]
#     )
# ]

# evaluate(test_cases, [
#     AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
#     FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
# ])